# Solutions · Chapter 03-08 · Loss, gradients, and how a model is fitted

Worked answers for `notebooks/03_math_foundations/03-08_loss_and_gradients.ipynb`.

The hand calculations are shown as arithmetic first and code second, because the point of E5 to E15 is
that you can do them without a computer. Run the setup cell, then work down.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# SYNTHETIC: the chapter's data, rebuilt
rng = np.random.default_rng(7)
n_days = 60
temperature = rng.uniform(10, 30, n_days)
rentals = 40 + 15 * temperature + rng.normal(0, 25, n_days)

mean_temp, sd_temp = temperature.mean(), temperature.std()
scaled_temp = (temperature - mean_temp) / sd_temp

exact_slope, exact_intercept = np.polyfit(temperature, rentals, 1)
best_slope, best_intercept = np.polyfit(scaled_temp, rentals, 1)
best_loss = np.mean((rentals - (best_intercept + best_slope * scaled_temp)) ** 2)


def loss(slope, intercept, feature=None):
    feature = temperature if feature is None else feature
    return float(np.mean((rentals - (intercept + slope * feature)) ** 2))


def derivative(function, at, step=1e-5):
    return (function(at + step) - function(at - step)) / (2 * step)


def fit(feature, learning_rate, steps):
    slope, intercept = 0.0, 0.0
    history = []
    for _ in range(steps):
        residual = (intercept + slope * feature) - rentals
        slope -= learning_rate * 2 * np.mean(residual * feature)
        intercept -= learning_rate * 2 * np.mean(residual)
        history.append(np.mean((rentals - (intercept + slope * feature)) ** 2))
    return slope, intercept, np.array(history)


print("set up. best achievable loss on the standardised feature: %.4f" % best_loss)

## Quick understanding

### E1

**One number.** That is the whole point of a loss function - it collapses 60 residuals into a single
score so that two candidate answers can be compared.

"Lower is better" says nothing about the *sign* of the residuals, because they are squared before they
are averaged. A model that is 10 too high on one day and 10 too low on the next has the same loss as
one that is 10 too low twice. Signed errors would cancel, and a badly wrong model that erred in both
directions equally would score zero.

### E2

`100 ** 6` = **1e+12** evaluations, or about eleven days at a microsecond each.

Grid search throws away the only useful thing each evaluation knows: **which way the ground tilts**.
It learns the height at a point and nothing about its neighbourhood, so every one of those 1e+12 points
has to be visited. A gradient costs about the same as one evaluation and tells you where to go next,
which turns an exhaustive search into a walk.

### E3

Both gradients are negative and the step is `parameter -= learning_rate * gradient`, so **both
parameters increase**.

The slope moves about **21.7 times further** than the intercept for the same learning rate, because
4185 / 193 = 21.7. That asymmetry is the narrow valley of the second failure lab, seen as two numbers
rather than as a picture.

### E4

1. **The learning rate is too small.** Fix: increase it, by a factor of ten, and check that the curve
   still falls smoothly.
2. **Too few steps.** The rate is fine and the curve is descending normally; it just has not been given
   long enough. Fix: run more steps, or add a stopping rule based on the improvement per step rather
   than a fixed count.

The two look identical on a single curve, which is why the practical move is to run longer *and* try a
larger rate, and see which one changes the picture.

In [ ]:
print("E2: 100 ** 6 = %.0e evaluations, %.1f days at 1 microsecond each"
      % (100.0 ** 6, 100.0 ** 6 * 1e-6 / 86400))
print("E3: the slope moves %.1f times further than the intercept" % (4185 / 193))

## Hand calculation

### E5

Starting from slope 0 and intercept 0, every prediction is 0, so the residuals are `prediction - y`:

| row | `x` | `y` | prediction | residual |
|---|---|---|---|---|
| 1 | 1 | 3 | 0 | -3 |
| 2 | 2 | 5 | 0 | -5 |
| 3 | 3 | 7 | 0 | -7 |

**Loss** = `(9 + 25 + 49) / 3` = `83 / 3` = **27.6667**

**Gradient for the slope** = `2 x mean(residual x x)` = `2 x mean(-3, -10, -21)` = `2 x (-34/3)` =
**-22.6667**

**Gradient for the intercept** = `2 x mean(residual)` = `2 x (-5)` = **-10**

Both negative, so both parameters need to increase - which is right, since the true line is
`y = 1 + 2x` and we are starting from `0 + 0x`.

### E6

`slope = 0 - 0.1 x (-22.6667)` = **2.2667**
`intercept = 0 - 0.1 x (-10)` = **1.0**

New predictions: `3.2667`, `5.5333`, `7.8`. Residuals: `0.2667`, `0.5333`, `0.8`.

**Loss** = `(0.0711 + 0.2844 + 0.64) / 3` = **0.3319**

The loss fell from 27.6667 to 0.3319, a factor of **83**. One step covered almost the whole distance,
which is what a large gradient far from the bottom buys you.

Note the residuals all changed sign. The step overshot.

### E7

`gradient for the slope` = **2.4889**, `gradient for the intercept` = **1.0667** - now **positive**,
confirming the overshoot.

`slope = 2.2667 - 0.1 x 2.4889` = **2.0178**
`intercept = 1.0 - 0.1 x 1.0667` = **0.8933**
**Loss** = **0.005267**

The gradients have shrunk from -22.6667 and -10 to 2.4889 and 1.0667: roughly **nine times smaller**.
Nobody reduced the step size - the gradient did it, because the ground near the bottom of the bowl is
nearly flat. This is the self-limiting behaviour the chapter's derivative table showed as a column of
falling numbers.

### E8

`slope = 0 - 1.0 x (-22.6667)` = **22.6667**, `intercept = 0 - 1.0 x (-10)` = **10**.

Predictions: `32.67`, `55.33`, `78`. Residuals: `29.67`, `50.33`, `71`. **Loss = 2818.2**, against
27.6667 at the start - a hundred times *worse* than doing nothing.

The direction was correct. The distance was not. On the right-hand panel of the failure-lab picture,
this is the first arrow: it points downhill, crosses the bottom, and lands far up the opposite wall.
The next gradient will be larger still, so a second step makes it worse again.

### E9

Because each step multiplies the remaining error by a factor - here roughly a fixed fraction, as the
chapter measured at exactly 0.9 - and a positive fraction repeated any number of times never reaches
zero. Gradient descent approaches the answer geometrically and arrives only in the limit.

In practice nobody wants exactness anyway. You stop when one of these is true: the improvement per step
drops below a tolerance, a fixed budget of steps is used up, or - the one that matters in real
modelling - the error on held-out data stops improving, which usually happens long before the training
loss flattens.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([3.0, 5.0, 7.0])


def tiny_step(slope, intercept, learning_rate):
    residual = (intercept + slope * x) - y
    return (slope - learning_rate * 2 * np.mean(residual * x),
            intercept - learning_rate * 2 * np.mean(residual),
            2 * np.mean(residual * x), 2 * np.mean(residual))


def tiny_loss(slope, intercept):
    return float(np.mean((y - (intercept + slope * x)) ** 2))


print("E5  loss %.4f   gradients %.4f and %.4f"
      % (tiny_loss(0, 0), 2 * np.mean((0 - y) * x), 2 * np.mean(0 - y)))

slope, intercept, gw, gb = tiny_step(0.0, 0.0, 0.1)
print("E6  slope %.4f  intercept %.4f  loss %.6f  (fell by a factor of %.1f)"
      % (slope, intercept, tiny_loss(slope, intercept), tiny_loss(0, 0) / tiny_loss(slope, intercept)))

slope2, intercept2, gw2, gb2 = tiny_step(slope, intercept, 0.1)
print("E7  gradients now %.4f and %.4f -> slope %.4f  intercept %.4f  loss %.6f"
      % (gw2, gb2, slope2, intercept2, tiny_loss(slope2, intercept2)))

big_slope, big_intercept, _, _ = tiny_step(0.0, 0.0, 1.0)
print("E8  at rate 1.0: slope %.4f  intercept %.4f  loss %.1f  (was %.4f)"
      % (big_slope, big_intercept, tiny_loss(big_slope, big_intercept), tiny_loss(0, 0)))

### E10

`f(3.5) = 12.25`, `f(2.5) = 6.25`, `f(3) = 9`.

- **Central:** `(12.25 - 6.25) / 1.0` = **6.0** - exactly right.
- **Forward:** `(12.25 - 9) / 0.5` = **6.5** - out by 8%.

The central difference is exact and it is not luck. Stepping the same distance either way, the curvature
bends the function up by the same amount on both sides, so when you subtract, the two curvature errors
cancel. For a parabola that cancellation is perfect at *any* step size. The forward difference has
nothing to cancel against, so the curvature is left in the answer.

### E11

`g(2.5) = 15.625`, `g(1.5) = 3.375`, `g(2) = 8`, `g(2.5) - g(2) = 7.625`.

- **Central:** `(15.625 - 3.375) / 1.0` = **12.25** - out by 2.1%.
- **Forward:** `7.625 / 0.5` = **15.25** - out by 27%.

The cubic curves *differently* on the two sides, so the cancellation is no longer perfect - but it is
still most of the way there. The central difference beats the forward one by more than a factor of ten
here, and the gap widens as the step shrinks: halve `h` and the central error falls by four while the
forward error only halves.

**So: always use the central difference when checking a gradient.** It costs one extra evaluation and
buys an order of magnitude of accuracy, which matters when you are trying to decide whether a 1e-4
disagreement is a bug or arithmetic.

In [ ]:
rows = []
for name, f, at, truth in [("x^2 at 3", lambda v: v ** 2, 3.0, 6.0),
                           ("x^3 at 2", lambda v: v ** 3, 2.0, 12.0)]:
    for h in [1.0, 0.5, 0.25, 0.1]:
        central = (f(at + h) - f(at - h)) / (2 * h)
        forward = (f(at + h) - f(at)) / h
        rows.append({"function": name, "h": h,
                     "central": round(central, 6), "central error": round(central - truth, 6),
                     "forward": round(forward, 6), "forward error": round(forward - truth, 6)})
print(pd.DataFrame(rows).to_string(index=False))

Read the two error columns down the `x^3` block: the central error goes 0.25, 0.0625, 0.015625, 0.0025 -
falling by four each time the step halves - while the forward error merely halves. That is the whole
argument for the extra evaluation.

### E12

The derivative is a straight line in the slope, so from -12,681.61 rising at 849.61 per unit it reaches
zero after

`12681.61 / 849.61` = **14.926**

which is the bottom, found from one derivative and one division.

For a million parameters the equivalent object is not a number but the matrix of second derivatives,
`p x p`. At `p = 1,000,000` that is **1e+12 entries**, which is **8 terabytes** at eight bytes each -
before anyone tries to invert it, which costs about `p^3` = 1e+18 operations. The gradient for the same
model is a list of a million numbers, 8 megabytes. That difference, not accuracy, is why the whole field
uses the slower method.

### E13

`log(0.01 / 311.5256) / log(0.9)` = **98.2**, so **99 further steps** - 100 in total from the start.

Two things worth noticing. It is *not* proportional to how close you want to get: each factor of ten
costs the same 22 steps, however far in you already are. And the data never entered the calculation -
only the learning rate did.

### E14

The remaining error is multiplied by `1 - 2 x learning_rate` each step, so:

- **0.5 is fastest.** The factor is exactly **zero**, so a single step lands on the answer. This is
  Newton's method: on standardised data the curvature is 2, and `1 / 2` is precisely the step that
  dividing by the curvature would have chosen.
- It converges when `|1 - 2 x learning_rate| < 1`, which is **`0 < learning_rate < 1`**.
- Above 0.5 it still converges but oscillates, overshooting and coming back with the sign flipped each
  step.
- At exactly 1 it bounces between two points forever, and **above 1 it diverges**.

### E15

| run | verdict | why |
|---|---|---|
| A | **too small** | after 50 steps it has removed 9% of the loss |
| B | **too small** | moving faster than A, still nowhere near 431.78 |
| C | **well chosen** | 435.0 at step 50, within 1% of the best possible |
| D | **well chosen, and fastest** | already at 431.8 by step 1 |
| E | **diverging** | rising by four orders of magnitude, not falling |

Between the two that work, **D beats C**: it arrives in a single step, so it is the 0.5 of E14. C is the
chapter's 0.05, which needs about 100 steps to get there.

The trap is A and B. Both are behaving perfectly correctly - the loss falls every step, nothing warns
you, and the curve looks reasonable if you plot it alone without a reference line. **A monotonically
falling loss is not evidence of a good learning rate.** Only comparing against what is achievable, or
seeing the curve flatten, tells you that.

In [ ]:
print("E12 one jump lands at : %.4f" % (12681.61 / 849.61))
print("E12 a million-parameter curvature matrix: %.0e entries, %.1f terabytes"
      % (1e6 ** 2, 1e6 ** 2 * 8 / 1e12))
print("E13 further steps needed: %.1f -> %d" % (np.log(0.01 / 311.5256) / np.log(0.9),
                                                int(np.ceil(np.log(0.01 / 311.5256) / np.log(0.9)))))
print("E13 steps per factor of ten: %.1f" % (np.log(0.1) / np.log(0.9)))
print()
print("E14 contraction factor by learning rate:")
for rate in [0.05, 0.25, 0.5, 0.75, 0.99, 1.0, 1.05]:
    factor = 1 - 2 * rate
    print("  %.2f -> %+6.2f   %s" % (rate, factor,
          "converges" if abs(factor) < 1 else "bounces forever" if abs(factor) == 1 else "diverges"))

## Coding

### E16 - early stopping

In [ ]:
def fit_until(feature, learning_rate, tolerance=1e-9, limit=100_000):
    slope, intercept, previous = 0.0, 0.0, np.inf
    for step in range(1, limit + 1):
        residual = (intercept + slope * feature) - rentals
        slope -= learning_rate * 2 * np.mean(residual * feature)
        intercept -= learning_rate * 2 * np.mean(residual)
        current = np.mean((rentals - (intercept + slope * feature)) ** 2)
        if abs(previous - current) < tolerance:
            return slope, intercept, current, step
        previous = current
    return slope, intercept, current, limit


slope, intercept, final, stopped = fit_until(scaled_temp, 0.05)
print("stopped at step %d with loss %.6f  (best possible %.6f)" % (stopped, final, best_loss))
print("the chapter ran a fixed 400 steps, so %d of them changed nothing measurable" % (400 - stopped))

Note what the tolerance is measuring: **the improvement per step, not the distance to the answer.** You
cannot use the second one, because knowing it would mean already knowing the answer. That is a small
lesson with a long reach - every stopping rule in machine learning watches a proxy, and choosing the
right proxy is chapter 05-03's subject.

### E17 - any number of features

In [ ]:
def fit_matrix(X, target, learning_rate, steps):
    weights = np.zeros(X.shape[1])
    for _ in range(steps):
        residual = X @ weights - target
        weights -= learning_rate * 2 * (X.T @ residual) / len(target)
    return weights


noise_rng = np.random.default_rng(11)
columns = np.column_stack([scaled_temp, scaled_temp ** 2, noise_rng.normal(size=n_days)])
columns = (columns - columns.mean(axis=0)) / columns.std(axis=0)      # standardise every column
design = np.column_stack([np.ones(n_days), columns])                  # and add the intercept column

descended = fit_matrix(design, rentals, 0.05, 5000)
solved = np.linalg.lstsq(design, rentals, rcond=None)[0]

print("gradient descent :", np.round(descended, 4))
print("np.linalg.lstsq  :", np.round(solved, 4))
print("largest disagreement: %.2e" % np.abs(descended - solved).max())

The loop is unchanged apart from `X @ weights` replacing `intercept + slope * feature`, and
`X.T @ residual / n` replacing the two hand-written gradients. **That is the version that scales to a
million features**, and it is why 03-07 spent a chapter on shapes.

The `1` column carries the intercept, so it needs no special handling - and note it is deliberately
*not* standardised, since a column of ones has zero standard deviation and dividing by that produces
`nan`. Standardise the features, then add the ones column.

### E18 - checking a gradient

In [ ]:
def check_gradient(function, analytic, at, step=1e-5):
    numerical = (function(at + step) - function(at - step)) / (2 * step)
    scale = max(abs(numerical), abs(analytic), 1e-12)
    return numerical, analytic, abs(numerical - analytic) / scale


here = 10.0
residual_here = (40 + here * temperature) - rentals

correct = 2 * np.mean(residual_here * temperature)
broken = np.mean(residual_here * temperature)              # the factor of 2 dropped

for label, candidate in [("correct", correct), ("factor of 2 dropped", broken)]:
    numerical, analytic, relative = check_gradient(lambda s: loss(s, 40), candidate, here)
    print("%-20s numerical %12.4f  analytic %12.4f  relative difference %.2e  %s"
          % (label, numerical, analytic, relative, "PASS" if relative < 1e-6 else "FAIL"))

The relative difference, not the absolute one, is what to test. These gradients are in the thousands, so
an absolute gap of 0.01 would be excellent here and catastrophic on a loss of order 1e-6. A threshold
around **1e-6 relative** is the usual choice for float64.

Writing this five-line function before writing a model is one of the highest-return habits in the
field. A wrong gradient does not crash - **it trains, slowly and to the wrong answer**, and it can cost
days to find by any other route.

### E19 - absolute error instead of squared error

In [ ]:
def fit_absolute(feature, steps=20_000, initial_rate=2.0):
    slope, intercept = 0.0, 0.0
    for step in range(1, steps + 1):
        residual = (intercept + slope * feature) - rentals
        rate = initial_rate / np.sqrt(step)          # the sign gradient never shrinks, so the step must
        slope -= rate * np.mean(np.sign(residual) * feature)
        intercept -= rate * np.mean(np.sign(residual))
    return slope, intercept


abs_slope, abs_intercept = fit_absolute(scaled_temp)
abs_raw_slope = abs_slope / sd_temp
abs_raw_intercept = abs_intercept - abs_slope * mean_temp / sd_temp


def mean_absolute(slope, intercept):
    return float(np.mean(np.abs(rentals - (intercept + slope * temperature))))


table = pd.DataFrame([
    {"fitted on": "squared error", "slope": round(exact_slope, 4), "intercept": round(exact_intercept, 4),
     "mean squared error": round(loss(exact_slope, exact_intercept), 2),
     "mean absolute error": round(mean_absolute(exact_slope, exact_intercept), 4)},
    {"fitted on": "absolute error", "slope": round(abs_raw_slope, 4), "intercept": round(abs_raw_intercept, 4),
     "mean squared error": round(loss(abs_raw_slope, abs_raw_intercept), 2),
     "mean absolute error": round(mean_absolute(abs_raw_slope, abs_raw_intercept), 4)},
])
print(table.to_string(index=False))

**Each loss wins on its own measure and loses on the other**, which is the cleanest possible statement
that the loss is a choice rather than a detail. Neither line is "the fit"; they are the answers to two
different questions.

Which rows account for the gap? The ones furthest from the line.

In [ ]:
squared_residuals = rentals - (exact_intercept + exact_slope * temperature)
absolute_residuals = rentals - (abs_raw_intercept + abs_raw_slope * temperature)

worst = np.argsort(-np.abs(squared_residuals))[:3]
print("the three days furthest from the squared-error line:")
print("  temperature      :", np.round(temperature[worst], 2))
print("  residual, squared-error fit :", np.round(squared_residuals[worst], 2))
print("  residual, absolute-error fit:", np.round(absolute_residuals[worst], 2))
print()
print("median |residual|  squared-error fit %.3f, absolute-error fit %.3f"
      % (np.median(np.abs(squared_residuals)), np.median(np.abs(absolute_residuals))))

The absolute-error line is **further** from the worst days and **closer** to the typical one. Squaring
makes a residual of 57 count as much as thirteen residuals of 16, so the squared-error line is dragged
towards the outliers; absolute error lets them go. This is 03-07's L1-versus-L2 distinction turning up
as a modelling decision - and if the far-off days are data-entry errors you want the absolute-error
line, while if they are real demand spikes you probably do not.

### E20 - the largest workable learning rate

In [ ]:
def converges(rate, steps=4000):
    with np.errstate(over="ignore", invalid="ignore"):
        _, _, history = fit(scaled_temp, rate, steps)
    return bool(np.isfinite(history[-1]) and history[-1] < best_loss * 1.0001)


workable = [round(0.01 * k, 2) for k in range(1, 150) if converges(0.01 * k)]
print("converges for rates %.2f to %.2f" % (workable[0], workable[-1]))
print("first rate that fails: %.2f" % (workable[-1] + 0.01))
print()
print("mean(scaled_temp ** 2) = %.4f, so 1 / mean(...) = %.4f"
      % (np.mean(scaled_temp ** 2), 1 / np.mean(scaled_temp ** 2)))

The empirical cut-off sits just under **1.0**, exactly `1 / mean(scaled_temp ** 2)`.

That is not a coincidence and it is the point of the exercise. The curvature of this loss is
`2 x mean(feature ** 2)`; standardising makes `mean(feature ** 2)` equal to 1, so the curvature is 2, the
ideal step is `1 / 2`, and the method survives up to twice that. **Standardising does not merely help -
it tells you what the learning rate should be**, which is why 0.1 is a sane default on scaled data and
meaningless on raw data.

## Interpretation

### E21

They have not looked at **held-out data**. Training loss falling for 200 epochs is compatible with the
model getting steadily better and equally compatible with it memorising the training rows, and the two
are indistinguishable from that curve alone.

The check is to plot the validation loss on the same axes. The failure it catches is **overfitting**:
training loss falling while validation loss turns and rises. That turning point, not the flattening of
the training curve, is where the model should have stopped. Chapter 05-01 is the whole story.

### E22

Neither symptom sounds like a scaling problem, which is why this one wastes so much time:

1. **It never seems to finish.** Every learning rate they try either explodes immediately or leaves the
   loss visibly falling after every budget they are willing to run. There is no comfortable middle,
   because the workable band is narrow and sits at an unguessably small number.
2. **The answer will not sit still.** Run it twice with slightly different settings and the coefficients
   come out different, because the walk stops at a different point along a valley floor that is nearly
   flat. On real data this shows up as coefficients that flip sign between runs, and people conclude the
   feature is unimportant rather than that the fit never converged.

With a direct solver instead of a descent the same condition number reappears as numerical warnings and
coefficients with implausible magnitudes.

## Debugging

### E23

`+=` steps **uphill**. The loss rises from the very first step, monotonically and fast, and overflows to
`inf` within tens of steps - and because the gradient grows as you climb, it accelerates.

The tempting answer to the second half is "plot the loss curve, and look for oscillation." **That answer
is wrong, and this data disproves it in an unusually blunt way.** Run the sign error at a sensible 0.05
next to a merely-too-large 1.05, and the two loss curves are not similar - they are *identical to every
decimal place*.

In [ ]:
# direction = -1 is the correct downhill step; +1 is the sign bug
def explode(feature, learning_rate, direction, steps=30):
    slope, intercept, slopes, history = 0.0, 0.0, [], []
    with np.errstate(over="ignore", invalid="ignore"):
        for _ in range(steps):
            residual = (intercept + slope * feature) - rentals
            slope += direction * learning_rate * 2 * np.mean(residual * feature)
            intercept += direction * learning_rate * 2 * np.mean(residual)
            slopes.append(slope)
            history.append(np.mean((rentals - (intercept + slope * feature)) ** 2))
    return np.array(slopes), np.array(history)


large_slopes, too_large = explode(scaled_temp, 1.05, -1)      # correct direction, huge step
buggy_slopes, sign_error = explode(scaled_temp, 0.05, +1)     # sensible step, wrong direction

print("rate too large, first six losses:", np.round(too_large[:6], 1))
print("sign error,     first six losses:", np.round(sign_error[:6], 1))
print("the two loss curves are identical:", np.allclose(too_large, sign_error))

It is a coincidence of these two rates - `1 - 2(1.05)` is `-1.1` and `1 + 2(0.05)` is `+1.1`, so the two
runs grow the error by the same factor and therefore have the same loss at every step - but the lesson
survives the coincidence. **The loss is a function of the distance from the answer, and it does not
record which side you are on.**

What separates them is the **parameter**, not the loss.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))

left.plot(too_large, "o-", color="#D55E00", markersize=3, label="rate too large (1.05)")
left.plot(sign_error, "s--", color="#0072B2", markersize=3, label="sign error (0.05)")
left.set_yscale("log")
left.set_title("the loss curves: indistinguishable")
left.set_xlabel("step")
left.set_ylabel("loss (log scale)")
left.legend()

right.plot(large_slopes[:10], "o-", color="#D55E00", markersize=4, label="rate too large (1.05)")
right.plot(buggy_slopes[:10], "s--", color="#0072B2", markersize=4, label="sign error (0.05)")
right.axhline(best_slope, color="#000000", linewidth=0.8)
right.set_title("the slope: one flips sign, one never turns back")
right.set_xlabel("step")
right.set_ylabel("slope")
right.legend()

plt.tight_layout()
plt.show()

print("rate too large, first six slopes:", np.round(large_slopes[:6], 1))
print("sign error,     first six slopes:", np.round(buggy_slopes[:6], 1))

On the right the difference is unmissable. **Too large a rate flips the slope's sign every step** -
187, -19, 208, -41 - because each step crosses the bottom of the bowl and lands on the far side. **A
sign error walks steadily away in one direction** - -9, -19, -30, -41 - and never crosses anything,
because it was never heading for the bottom.

So the diagnosis is: plot the *parameters*, not only the loss. A monotone run away from a sensible
starting region means the direction is wrong; alternating signs mean the direction is right and the
distance is not. And the cheapest check of all remains E18 - a sign error in a gradient is exactly what
a numerical gradient check is for, and it would have caught this before a single step was taken.

### E24

1. **Divide the learning rate by ten** and re-run. This is the cause the large majority of the time, it
   takes one line, and if the `nan` disappears you are finished.
2. **Check the feature scales.** `print(X.mean(axis=0))` and `print(X.std(axis=0))`. A column in the
   millions next to one in the units will overflow at a rate the other columns need, and no single
   learning rate exists that suits both.
3. **Check the gradient against a numerical one** (E18), and check the data for `nan` or `inf` that was
   there before the fit started - `np.isfinite(X).all()`. A single missing value poisons every
   subsequent step, and the symptom is identical.

In that order, because the cost of checking rises steeply down the list and the likelihood falls.

## Exam and interview reasoning

### E25

> "You need three things: a way to make a prediction, a loss that scores how wrong it is on your data,
> and the gradient of that loss - which just says, for each parameter, whether nudging it up makes
> things better or worse. Then you start anywhere, take a small step against the gradient, and repeat.
> The steps get shorter on their own as you approach the answer, because the gradient shrinks there. The
> step size is the one thing you choose: too small and it never arrives, too large and it overshoots and
> diverges. Everything from linear regression to a large neural network is that loop - what changes is
> the prediction function and the loss, not the loop."

**The follow-up.** Newton's method converges in one step because it divides by the curvature instead of
guessing a step size - but the curvature of a model with `p` parameters is a `p x p` matrix that has to
be built and inverted, which is `p^2` memory and about `p^3` work. At a million parameters that is 8
terabytes and 1e+18 operations. A gradient is `p` numbers and one pass over the data. Gradient descent
is not the better algorithm; it is the one that fits in memory - which is also why the useful middle
ground (momentum, Adam, L-BFGS) consists of methods that *approximate* curvature without ever forming
the matrix.

## Transfer to a different situation

### E26

**What changes: the loss, and therefore the gradient.** Squared error on a probability punishes
confident-and-wrong far too gently and is not convex once a squashing function is involved, so you use
**log-loss** (cross-entropy), and you pass the linear output through a sigmoid to keep the prediction
between 0 and 1.

**What stays exactly the same:** the loop. Predict, measure the residual, form the gradient, step
against it, repeat. Remarkably, the gradient of log-loss with respect to the weights comes out as
`mean(residual x feature)` - the same shape as here, with `residual` now meaning `predicted probability
minus actual 0-or-1`. That is chapter 07-04, and the reason it will feel like a small change rather than
a new subject.

## Explain it to someone non-technical

### E27

> Imagine standing on a foggy hillside, trying to reach the lowest point. You cannot see the valley, but
> you can feel which way the ground slopes under your feet. So you take a step downhill, feel again, and
> step again. The ground flattens as you near the bottom, so your steps naturally get smaller and you
> settle rather than overshoot. That is all the computer does. The hill is "how wrong the line is", the
> position is the line it is currently drawing, and stepping downhill means adjusting the line so it
> fits the data a little better each time.

(89 words, and it contains the three things that matter: a measure of wrongness, local information only,
and repetition.)

## Optional challenge

### E28 - momentum

In [ ]:
def fit_momentum(feature, learning_rate, friction=0.9, tolerance=1e-6, limit=200_000):
    slope = intercept = velocity_slope = velocity_intercept = 0.0
    previous = np.inf
    for step in range(1, limit + 1):
        with np.errstate(over="ignore", invalid="ignore"):
            residual = (intercept + slope * feature) - rentals
            velocity_slope = friction * velocity_slope + 2 * np.mean(residual * feature)
            velocity_intercept = friction * velocity_intercept + 2 * np.mean(residual)
            slope -= learning_rate * velocity_slope
            intercept -= learning_rate * velocity_intercept
            current = np.mean((rentals - (intercept + slope * feature)) ** 2)
        if not np.isfinite(current):
            return None, step
        if abs(previous - current) < tolerance:
            return current, step
        previous = current
    return current, limit


rows = []
for rate in [0.0005, 0.0010, 0.0015]:
    final, step = fit_momentum(temperature, rate)
    rows.append({"learning rate": rate, "with momentum": step if final else "diverged",
                 "loss": round(final, 4) if final else None})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("without momentum, at 0.0015, the chapter measured 21,769 steps")
print("standardised and without momentum, 115")

Momentum turns 21,769 steps into a few thousand - a real improvement, and exactly the improvement it was
designed for. The velocity accumulates in the direction the walk keeps travelling, which is *along* the
valley floor, while the back-and-forth components across the valley cancel each other out from step to
step. It is the ball-rolling-downhill version of the same walk.

But read the last line of the output. **Standardising the feature, with no momentum and no cleverness at
all, still beats it by an order of magnitude.** Momentum makes a badly conditioned problem tolerable;
scaling makes it disappear.

That ordering is the practical lesson and it generalises well past this chapter: fix the geometry of the
problem first, then reach for a better optimiser. Modern optimisers like Adam are, in large part,
machinery for coping with badly scaled directions - which is why they are indispensable in deep networks,
where you cannot simply standardise the inputs to every layer once and be done. (Though people try. That
is what batch normalisation is.)